In [2]:
import math
import csv
import numpy as np

# ============================================================
# Класс нейросети
# ============================================================
class NeuralNetwork:
    def __init__(self, hiddenWeights, outputWeights, hiddenBias, outputBias, requestedEra=1000, lr=0.01, momentum=0.1):
        self.requestedEra = requestedEra
        self.hiddenWeights = hiddenWeights
        self.outputWeights = outputWeights
        self.hiddenBias = hiddenBias
        self.outputBias = outputBias
        self.lr = lr
        self.momentum = momentum

        self.dataset = []
        self.answersList = []

        # Для хранения дельт
        self.hiddenWeightDeltas = np.zeros_like(hiddenWeights)
        self.outputWeightDeltas = np.zeros_like(outputWeights)
        self.hiddenBiasDeltas = np.zeros_like(hiddenBias)
        self.outputBiasDelta = 0.0

    # ------------------------------------------------------------
    # Чтение данных
    # ------------------------------------------------------------
    def readFromCsv(self, datasetPath: str):
        with open(datasetPath, 'r') as csvDataset:
            reader = csv.reader(csvDataset, delimiter='\t')
            next(reader)  # пропускаем заголовок
            for row in reader:
                a, b, c, d = map(float, row)
                self.dataset.append([a, b, c, d])
        self._normalizeDataset()
        print("Нормализация входов выполнена.")
    #новая функция нормализации
    def _normalizeDataset(self):
        data = np.array(self.dataset)
        inputs = data[:, :3]
        outputs = data[:, 3]

        # Z-score нормализация входов
        self.input_means = inputs.mean(axis=0)
        self.input_stds = inputs.std(axis=0)
        self.input_stds[self.input_stds == 0] = 1e-6
        self.normalized_inputs = (inputs - self.input_means) / self.input_stds

        # Масштабирование выхода в [-1, 1] для стабильности
        self.output_scale = np.max(np.abs(outputs))
        self.normalized_outputs = outputs / self.output_scale

        self.normalized_dataset = np.hstack((self.normalized_inputs, self.normalized_outputs.reshape(-1, 1)))

    # новая функция денормализации
    def denormalize_output(self, y):
        return y * self.output_scale

    # ------------------------------------------------------------
    # Прямое распространение
    # ------------------------------------------------------------
    #Биасы добавлены + выходной нейрон изменил
    def forward(self, x):
        hidden_input = np.dot(self.hiddenWeights, x) + self.hiddenBias
        hidden_output = np.tanh(hidden_input)
        output_input = np.dot(self.outputWeights, hidden_output) + self.outputBias
        output = output_input  # линейный выход
        return hidden_output, output

    # ------------------------------------------------------------
    # Обратное распространение
    # ------------------------------------------------------------
    def backward(self, x, hidden_output, output, target):
        error = target - output
        d_output = error  # производная линейной функции = 1

        d_hidden = (1 - hidden_output ** 2) * (self.outputWeights * d_output)

        # обновляем веса с моментом
        output_deltas = self.lr * d_output * hidden_output + self.momentum * self.outputWeightDeltas
        hidden_deltas = self.lr * np.outer(d_hidden, x) + self.momentum * self.hiddenWeightDeltas

        # обновляем смещения
        output_bias_delta = self.lr * d_output + self.momentum * self.outputBiasDelta
        hidden_bias_deltas = self.lr * d_hidden + self.momentum * self.hiddenBiasDeltas

        # применяем обновления
        self.outputWeights += output_deltas
        self.hiddenWeights += hidden_deltas
        self.outputBias += output_bias_delta
        self.hiddenBias += hidden_bias_deltas

        # сохраняем дельты для момента
        self.outputWeightDeltas = output_deltas
        self.hiddenWeightDeltas = hidden_deltas
        self.outputBiasDelta = output_bias_delta
        self.hiddenBiasDeltas = hidden_bias_deltas

        return error ** 2

    # ------------------------------------------------------------
    # Обучение
    # ------------------------------------------------------------
    #Mse считается на нормализованных выходах
    def train(self):
        for era in range(self.requestedEra):
            mse = 0.0
            np.random.shuffle(self.normalized_dataset)
            for sample in self.normalized_dataset:
                x = sample[:3]
                target = sample[3]
                hidden_output, output = self.forward(x)
                mse += self.backward(x, hidden_output, output, target)
            mse /= len(self.normalized_dataset)
            if era % 100 == 0:
                print(f"Эпоха {era:5d}, MSE: {mse:.6f}, LR: {self.lr:.4f}")
            if mse < 1e-5:
                print(f"Обучение завершено на эпохе {era}")
                break

    # ------------------------------------------------------------
    # Предсказание
    # ------------------------------------------------------------
    #Для денормализации(тесты и валидация)
    def predict(self, a, b, c):
        x = np.array([a, b, c])
        x_norm = (x - self.input_means) / self.input_stds
        _, y_norm = self.forward(x_norm)
        return self.denormalize_output(y_norm)

In [12]:
 hiddenWeights = np.array([
    [0.1, 0.9, 0.3],
    [0.4, 0.7, 0.1],
    [0.3, 0.2, 0.2]
])
outputWeights = np.array([0.5, 0.4, 0.6])
hiddenBias = np.array([0.1, 0.1, 0.1])
outputBias = 0.1

print("Начальные веса:")
print("Hidden:", hiddenWeights)
print("Output:", outputWeights)
print("Bias H:", hiddenBias, "Bias O:", outputBias)

myAI = NeuralNetwork(hiddenWeights, outputWeights, hiddenBias, outputBias, requestedEra=5000)
myAI.readFromCsv("discriminant_data_3.csv")
myAI.train()

Начальные веса:
Hidden: [[0.1 0.9 0.3]
 [0.4 0.7 0.1]
 [0.3 0.2 0.2]]
Output: [0.5 0.4 0.6]
Bias H: [0.1 0.1 0.1] Bias O: 0.1
Нормализация входов выполнена.
Эпоха     0, MSE: 0.063796, LR: 0.0100
Эпоха   100, MSE: 0.002198, LR: 0.0100
Эпоха   200, MSE: 0.002135, LR: 0.0100
Эпоха   300, MSE: 0.002128, LR: 0.0100
Эпоха   400, MSE: 0.002120, LR: 0.0100
Эпоха   500, MSE: 0.002121, LR: 0.0100
Эпоха   600, MSE: 0.002118, LR: 0.0100
Эпоха   700, MSE: 0.002106, LR: 0.0100
Эпоха   800, MSE: 0.002102, LR: 0.0100
Эпоха   900, MSE: 0.002092, LR: 0.0100
Эпоха  1000, MSE: 0.001980, LR: 0.0100
Эпоха  1100, MSE: 0.000433, LR: 0.0100
Эпоха  1200, MSE: 0.000309, LR: 0.0100
Эпоха  1300, MSE: 0.000238, LR: 0.0100
Эпоха  1400, MSE: 0.000155, LR: 0.0100
Эпоха  1500, MSE: 0.000108, LR: 0.0100
Эпоха  1600, MSE: 0.000098, LR: 0.0100
Эпоха  1700, MSE: 0.000093, LR: 0.0100
Эпоха  1800, MSE: 0.000089, LR: 0.0100
Эпоха  1900, MSE: 0.000088, LR: 0.0100
Эпоха  2000, MSE: 0.000084, LR: 0.0100
Эпоха  2100, MSE: 0.0000

In [13]:
print("\nТестирование:")
    # test_cases = [
    #     (1, 2, 1),
    #     (1, 5, 1),
    #     (2, 4, 1),
    #     (0.5, 3, 2),
    #     (4, 4, 4)
    # ]

test_cases = [
    (5.61478811062392, 68.47622120462377, 36.24445585206049),
    (69.9002065933079, 24.549362104375323, 14.301768725511709),
    (5.569494176827933, 83.73941658515918, 70.18158589106298),
    (3.4891985338090854, 12.212755486160841, 66.48070854382573),
    (48.54349226782931, 16.09995891170176, 21.474717895489515)
]
for a, b, c in test_cases:
    pred = myAI.predict(a, b, c)
    actual = b**2 - 4*a*c
    print(f"a={a}, b={b}, c={c} → предсказано: {pred:.3f}, реально: {actual:.3f}, ошибка: {abs(pred-actual):.3f}")


Тестирование:
a=5.61478811062392, b=68.47622120462377, c=36.24445585206049 → предсказано: 3781.578, реально: 3874.973, ошибка: 93.395
a=69.9002065933079, b=24.549362104375323, c=14.301768725511709 → предсказано: -3814.989, реально: -3396.115, ошибка: 418.874
a=5.569494176827933, b=83.73941658515918, c=70.18158589106298 → предсказано: 5884.772, реально: 5448.786, ошибка: 435.985
a=3.4891985338090854, b=12.212755486160841, c=66.48070854382573 → предсказано: -1533.648, реально: -778.706, ошибка: 754.942
a=48.54349226782931, b=16.09995891170176, c=21.474717895489515 → предсказано: -4055.394, реально: -3910.623, ошибка: 144.772


In [14]:
test_cases = [
    (57.54220842403881, 48.08074971868062, 58.67559278925581),
    (96.48778776312248, 70.10696726438404, 5.217940798543723),
    (23.916318080321435, 18.886379636110057, 47.47703885686927),
    (31.109609617485827, 67.47097267795215, 47.53945366732694),
    (6.122051188268588, 62.60487741460959, 12.06462209832774),
    (75.17138770990009, 0.3152453868159874, 92.73139768197484),
    (23.234280339010255, 19.485294562730076, 5.284185376404185),
    (52.62747365149391, 39.34826023659893, 46.42300752354182),
    (47.99008487118863, 58.51591459538526, 28.713187586822137),
    (72.99655347143035, 53.94709284287125, 67.03389364937374),
    (26.357361607481902, 17.52611250490423, 80.4970748922207),
    (64.59948538087295, 46.232154838027895, 4.748519541384409),
    (65.89314148081482, 88.73075053968701, 26.16903431679905),
    (62.90618054466393, 31.34035669175865, 56.53050795841362),
    (33.90803268510974, 81.9215350716808, 6.849241225429214),
    (71.10312768733675, 21.630656410981867, 70.61015099225155),
    (70.7048047118774, 35.928155665362254, 11.059994201618688),
    (25.227968848369784, 27.038698650224227, 35.63349458283558),
    (48.187493670777414, 59.147588668348895, 83.13851521201435),
    (25.033888531276595, 89.55615495513331, 19.183847003688374),
    (5.61478811062392, 68.47622120462377, 36.24445585206049),
    (69.9002065933079, 24.549362104375323, 14.301768725511709),
    (5.569494176827933, 83.73941658515918, 70.18158589106298),
    (3.4891985338090854, 12.212755486160841, 66.48070854382573),
    (48.54349226782931, 16.09995891170176, 21.474717895489515),
    (78.84193932288129, 52.508138007343895, 82.2616093143445),
    (87.44181422847433, 30.600004001806074, 24.900617506938104),
    (70.04197165825377, 55.7745001273916, 18.739836849312113),
    (33.535277962018434, 79.79289990920513, 27.755134448983636),
    (95.23752176940506, 6.6208217082220395, 53.675246244460034),
    (27.55089997299356, 54.459524190034315, 79.60894461817549),
    (31.37756890999414, 40.032023673298234, 7.109381961019749),
    (11.63710553867307, 89.34718811237784, 14.19878660658356),
    (53.74655559331153, 80.13975056412808, 36.2222073155799),
    (25.349489021933895, 24.77064233809988, 87.06694304175033),
    (43.99438237454393, 85.58327828663495, 7.621967231610929),
    (70.54329431969734, 33.86908256486521, 61.95707462168415),
    (58.81485520956646, 90.10502655439198, 19.015959577673414),
    (83.24790955404427, 40.40764040835491, 42.04666533572745),
    (20.471021401540906, 32.83286279445454, 62.55850732603938),
    (35.57894617412073, 12.923458297316262, 68.88849847492888),
    (77.54396998170839, 19.318509975825624, 27.033939073522628),
    (72.6632481169159, 34.79799487977172, 55.83761441989592),
    (7.304350783608016, 52.3954876020586, 48.04298033905115),
    (44.91515860505851, 82.10528288273211, 52.82409123015429),
    (8.119920527325416, 21.981764500927508, 68.32266272010125),
    (55.89463469527622, 44.22908113851409, 13.508909515288536),
    (51.88564628069232, 85.07291334901413, 11.477817405200218),
    (81.85506021349995, 37.40519343182377, 1.6300174384272847),
    (64.52716765224056, 98.29413376534421, 80.28637288470068),
    (17.035936639326277, 68.47666933440637, 48.46251101253847),
    (39.25470491785031, 84.2489448940387, 36.81733127344),
    (52.68720855793976, 63.65841058283333, 83.99169205483709),
    (43.2108227351842, 33.47907324672287, 35.76677371383732),
    (26.412087563238572, 8.92478845994916, 44.84752642485397),
    (13.388993423612227, 59.05612618066384, 93.42552953000701),
    (63.36936767967407, 60.36887318316578, 46.040058395133826),
    (55.86308059244472, 69.71600738702286, 18.828007957292364),
    (85.46637375908978, 22.088406393653244, 60.75929334765959),
    (64.46184408063827, 47.62627120313014, 8.286373905196724),
    (18.23913677607385, 68.45936844889026, 7.616899820945196)
]

for a, b, c in test_cases:
    pred = myAI.predict(a, b, c)
    actual = b**2 - 4*a*c
    print(f"a={a}, b={b}, c={c} → предсказано: {pred:.3f},реально: {actual:.3f}, ошибка: {abs(pred-actual):.3f}")

a=57.54220842403881, b=48.08074971868062, c=58.67559278925581 → предсказано: -11292.110,реально: -11193.534, ошибка: 98.576
a=96.48778776312248, b=70.10696726438404, c=5.217940798543723 → предсказано: 2960.837,реально: 2901.117, ошибка: 59.721
a=23.916318080321435, b=18.886379636110057, c=47.47703885686927 → предсказано: -4204.084,реально: -4185.209, ошибка: 18.876
a=31.109609617485827, b=67.47097267795215, c=47.53945366732694 → предсказано: -1488.254,реально: -1363.403, ошибка: 124.850
a=6.122051188268588, b=62.60487741460959, c=12.06462209832774 → предсказано: 3799.180,реально: 3623.930, ошибка: 175.250
a=75.17138770990009, b=0.3152453868159874, c=92.73139768197484 → предсказано: -28076.744,реально: -27882.892, ошибка: 193.852
a=23.234280339010255, b=19.485294562730076, c=5.284185376404185 → предсказано: 39.859,реально: -111.420, ошибка: 151.280
a=52.62747365149391, b=39.34826023659893, c=46.42300752354182 → предсказано: -8169.743,реально: -8224.217, ошибка: 54.474
a=47.9900848711886